In [2]:
import numpy as np
import pandas as pd
from pathlib import Path


OUT_DIR = Path("../outputs")
MONTHS_SAVED = 12      # months of revenue kept when a customer is saved (ASSUMPTION)
OFFER_COST = 30        # cost of one retention offer, in $ (ASSUMPTION)
SUCCESS_RATE = 0.30    # share of contacted churners the offer saves (ASSUMPTION)
# ------------------

pred = pd.read_csv(OUT_DIR / "churn_predictions.csv")
truth = pd.read_csv(OUT_DIR / "customers_cleaned.csv",
                    usecols=["customerID", "Churn", "MonthlyCharges"])
d = pred.merge(truth, on="customerID")
d["actual"] = (d["Churn"] == "Yes").astype(int)
print(len(d), "held-out customers")          # expect 1409

def evaluate(threshold, success=SUCCESS_RATE, cost=OFFER_COST):
    contacted = d[d["churn_probability"] >= threshold]
    churners = contacted[contacted["actual"] == 1]
    saved_value = (churners["MonthlyCharges"] * MONTHS_SAVED * success).sum()
    spend = len(contacted) * cost
    return {
        "threshold": round(threshold, 2),
        "contacted": len(contacted),
        "churners_reached": len(churners),
        "recall": round(len(churners) / d["actual"].sum(), 3),
        "precision": round(len(churners) / max(len(contacted), 1), 3),
        "net_benefit": round(saved_value - spend),
    }

table = pd.DataFrame([evaluate(t) for t in np.arange(0.10, 0.91, 0.05)])
print(table.sort_values("net_benefit", ascending=False).head(8).to_string(index=False))

best = table.loc[table["net_benefit"].idxmax()]
print("\nBest threshold:", best["threshold"], "| net benefit:", best["net_benefit"])

1409 held-out customers
 threshold  contacted  churners_reached  recall  precision  net_benefit
      0.30        808               347   0.928      0.429        66625
      0.20        927               360   0.963      0.388        66423
      0.35        756               338   0.904      0.447        66418
      0.40        694               324   0.866      0.467        65940
      0.25        863               351   0.939      0.407        65879
      0.15        993               365   0.976      0.368        65780
      0.45        649               314   0.840      0.484        65318
      0.10       1083               370   0.989      0.342        64653

Best threshold: 0.3 | net benefit: 66625.0


In [3]:
rows = []
for success in [0.10, 0.20, 0.30, 0.50]:
    for cost in [15, 30, 60]:
        t = pd.DataFrame([evaluate(x, success, cost) for x in np.arange(0.10, 0.91, 0.05)])
        top = t.loc[t["net_benefit"].idxmax()]
        rows.append({"success_rate": success, "offer_cost": cost,
                     "best_threshold": top["threshold"],
                     "net_benefit": top["net_benefit"]})
print(pd.DataFrame(rows).to_string(index=False))

 success_rate  offer_cost  best_threshold  net_benefit
          0.1          15            0.45      18528.0
          0.1          30            0.70       9989.0
          0.1          60            0.80       1787.0
          0.2          15            0.20      48917.0
          0.2          30            0.45      37056.0
          0.2          60            0.70      19979.0
          0.3          15            0.10      80898.0
          0.3          30            0.30      66625.0
          0.3          60            0.45      45848.0
          0.5          15            0.10     145660.0
          0.5          30            0.15     129493.0
          0.5          60            0.35     103137.0
